In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import struct
import os

# --- 1. Configurações e Preparação de Dados ---
BATCH_SIZE = 64
LEARNING_RATE = 0.01
EPOCHS = 15

# Transformação: Converte para Tensor e normaliza (importante para convergência)
transform = transforms.Compose([
    transforms.ToTensor(),
    #transforms.Normalize((0.5,), (0.5,))
])

print("Baixando e carregando MNIST...")
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# --- 2. Definição da Arquitetura MLP ---
class MNIST_MLP(nn.Module):
    def __init__(self):
        super(MNIST_MLP, self).__init__()
        # Topologia: 784 -> 64 -> 32 -> 10
        self.flatten = nn.Flatten()

        # Camada 1: 784 entradas -> 64 saídas
        self.fc1 = nn.Linear(784, 64)

        # Camada 2: 64 entradas -> 32 saídas
        self.fc2 = nn.Linear(64, 32)

        # Camada 3: 32 entradas -> 10 saídas
        self.fc3 = nn.Linear(32, 10)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Instanciar modelo, perda e otimizador
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MNIST_MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --- 3. Treinamento ---
print(f"Iniciando treinamento em: {device}")
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Época [{epoch+1}/{EPOCHS}], Loss: {running_loss/len(train_loader):.4f}")

print("Treinamento concluído.")

# --- 4. Função de Exportação para Hexadecimal ---
def float_to_fixed_hex(f, width=32, p_int=10):
    """
    Converte float para representação HEX em Ponto Fixo (Complemento de 2).
    Width: 32, P_Int: 10 -> Frac: 22 bits
    """
    frac_bits = width - p_int
    scale = 2 ** frac_bits

    # 1. Escalar o valor (Float -> Fixed Point Integer)
    f_scaled = int(round(f * scale))

    # 2. Lidar com limites (Saturação simples para evitar wrap-around destrutivo)
    max_val = (2 ** (width - 1)) - 1
    min_val = -(2 ** (width - 1))
    if f_scaled > max_val: f_scaled = max_val
    if f_scaled < min_val: f_scaled = min_val

    # 3. Converter para representação de bit (lidando com negativos via máscara)
    # Ex: -1 em 32 bits -> 0xFFFFFFFF
    hex_val = hex(f_scaled & ((1 << width) - 1))[2:].zfill(width // 4)

    return hex_val

def export_tensor_to_mem(tensor, filename):
    data = tensor.detach().cpu().numpy().flatten()
    with open(filename, 'w') as f:
        for value in data:
            # USA A NOVA FUNÇÃO AQUI
            hex_val = float_to_fixed_hex(value, width=32, p_int=10)
            f.write(f"{hex_val}\n")
    print(f"Arquivo gerado: {filename} | Elementos: {len(data)}")

# --- 5. Exportação dos Pesos e Bias ---
print("\nExportando arquivos .mem...")

# Acessando os pesos e biases do modelo
# PyTorch armazena pesos como (Out_Features, In_Features)
# Flatten irá linearizar isso.

# Camada 1
export_tensor_to_mem(model.fc1.weight, 'H1W.mem')
export_tensor_to_mem(model.fc1.bias,   'H1B.mem')

# Camada 2
export_tensor_to_mem(model.fc2.weight, 'H2W.mem')
export_tensor_to_mem(model.fc2.bias,   'H2B.mem')

# Camada 3
export_tensor_to_mem(model.fc3.weight, 'H3W.mem')
export_tensor_to_mem(model.fc3.bias,   'H3B.mem')

print("\nProcesso finalizado com sucesso.")

Baixando e carregando MNIST...


100%|██████████| 9.91M/9.91M [00:00<00:00, 16.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 465kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.30MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.85MB/s]


Iniciando treinamento em: cpu
Época [1/15], Loss: 0.2588
Época [2/15], Loss: 0.1448
Época [3/15], Loss: 0.1254
Época [4/15], Loss: 0.1168
Época [5/15], Loss: 0.1057
Época [6/15], Loss: 0.1016
Época [7/15], Loss: 0.0933
Época [8/15], Loss: 0.0866
Época [9/15], Loss: 0.0816
Época [10/15], Loss: 0.0850
Época [11/15], Loss: 0.0744
Época [12/15], Loss: 0.0775
Época [13/15], Loss: 0.0731
Época [14/15], Loss: 0.0758
Época [15/15], Loss: 0.0705
Treinamento concluído.

Exportando arquivos .mem...
Arquivo gerado: H1W.mem | Elementos: 50176
Arquivo gerado: H1B.mem | Elementos: 64
Arquivo gerado: H2W.mem | Elementos: 2048
Arquivo gerado: H2B.mem | Elementos: 32
Arquivo gerado: H3W.mem | Elementos: 320
Arquivo gerado: H3B.mem | Elementos: 10

Processo finalizado com sucesso.


In [ ]:
import matplotlib.pyplot as plt

NUM_SAMPLES = 10000
OUTPUT_HEX_FILE = "test_samples.mem"
OUTPUT_LABEL_FILE = "test_labels.mem"

# Transformação: Deve ser IDÊNTICA à usada no treinamento
# Se o treino normalizou com (0.5, 0.5), o teste DEVE seguir o mesmo padrão
# para que os valores de ponto flutuante façam sentido para a rede.
transform = transforms.Compose([
    transforms.ToTensor(),
    #transforms.Normalize((0.5,), (0.5,))
])

print("Carregando MNIST (Test Set)...")
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print(f"Extraindo {NUM_SAMPLES} amostras...")

hex_data_buffer = []
labels_buffer = []
images_for_plot = []

# Iterar sobre as primeiras N amostras
for i in range(NUM_SAMPLES):
    image_tensor, label = test_dataset[i]

    # Salvar label
    labels_buffer.append(label)

    # Processar imagem para Hex
    # Flatten: (1, 28, 28) -> (784)
    flat_data = image_tensor.flatten().numpy()

    # Converter cada pixel normalizado para hex
    for pixel_val in flat_data:
        hex_data_buffer.append(float_to_fixed_hex(pixel_val))

    # Guardar imagem original para plotagem (desnormalizando para visualização)
    # A normalização foi (x - 0.5) / 0.5, então volta com x * 0.5 + 0.5
    #img_display = image_tensor.numpy().squeeze() * 0.5 + 0.5
    img_display = image_tensor.numpy().squeeze()
    images_for_plot.append(img_display)

# Escrever arquivo .mem (Todos os pixels de todas as 10 imagens em sequência)
# Total de linhas = 10 * 784 = 7840 linhas
with open(OUTPUT_HEX_FILE, 'w') as f:
    for hex_val in hex_data_buffer:
        f.write(f"{hex_val}\n")
print(f"Arquivo gerado: {OUTPUT_HEX_FILE} (Inputs para simulação)")

# Escrever gabarito (labels)
with open(OUTPUT_LABEL_FILE, 'w') as f:
    for _, label in enumerate(labels_buffer):
        f.write(f"{hex(label)}\n")
print(f"Arquivo gerado: {OUTPUT_LABEL_FILE} (Gabarito)")

print("\nProcesso concluído.")

Carregando MNIST (Test Set)...
Extraindo 10000 amostras...
Arquivo gerado: test_samples.mem (Inputs para simulação)
Arquivo gerado: test_labels.mem (Gabarito)

Processo concluído.


In [ ]:
# --- 6. Verificação de Precisão do Modelo ---
print("\nIniciando avaliação de precisão...")

model.eval()  # Define o modelo para modo de avaliação
correct = 0
total = 0

# Usamos o DataLoader de teste para facilitar a iteração em batches
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad(): # Desabilita o cálculo de gradientes para economizar memória e processamento
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # Predição
        outputs = model(images)

        # A classe escolhida é a que tiver o maior valor (Logit)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Acurácia final no conjunto de teste (10.000 imagens): {accuracy:.2e}%")

# Verificação específica das N amostras exportadas para o arquivo .mem
print(f"Validando as primeiras {NUM_SAMPLES} amostras (as que foram exportadas)...")
sample_correct = 0
with torch.no_grad():
    for i in range(NUM_SAMPLES):
        img_tensor, label = test_dataset[i]
        img_tensor = img_tensor.to(device).unsqueeze(0) # Adiciona dimensão de batch (1, 1, 28, 28)

        output = model(img_tensor)
        _, pred = torch.max(output, 1)

        if pred.item() == label:
            sample_correct += 1

print(f"Acurácia nas {NUM_SAMPLES} amostras exportadas: {(100 * sample_correct / NUM_SAMPLES):.2f}%")


Iniciando avaliação de precisão...
Acurácia final no conjunto de teste (10.000 imagens): 9.69e+01%
Validando as primeiras 10000 amostras (as que foram exportadas)...
Acurácia nas 10000 amostras exportadas: 96.93%
